In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Receiver-Independent-Source-V1: fixed Milestone1B

One prospectively fixed new prompt and seed, after the receiver rules were frozen. Generate exactly one PAYLOAD_MULTI 50-step trajectory and one native decode, persist original RGB+SHA, then exit the source process. A separate process performs one M05 encode/write/decode and one full libx264 CRF18 yuv420p 8fps MP4 save/probe/readback. No extra P0/P1/M1/quality arms, old saved source, generation retry or selection.

Fixed prompt: locked camera, a small blue toy car slowly rolling left to right across a wooden tabletop, steady soft daylight, no people, no cuts; seed=2026100601.

Writer legally uses message/key/prompt; crop preparation uses starts. The blind receiver receives only current RGB, public protocol and its own key. Writer evidence, crop truth and message do not select scores/estimates/plans/alignment. Evaluation POSTHOC config first reads after payload seal.

Clone [2:179], [3:180], [38:127], [39:128] from this run full MP4 readback, with no new codec. Four fresh framewise encodes; eight scores; 392 candidates / 9456 locals. Complete finite unique estimates drive p=offset%4; no fallback or BER choice. R44/R22 and original Counter/coordinate/strength/codebook semantics stay fixed. Sync seal precedes the physical plan; payload seal precedes posthoc.

Sixteen logical slots retain 506880 votes / 16896 time-bit rows / 512 final bits; Wan physical upper bounds 12 encodes / 16 reads. Per-observation phase sharing and p0 aliases are preserved. Missing/source/codec/interruption failures remain. Exact offset, phase, final bits and time/channel margins are separate; all-time-bit-positive is descriptive, not a new PASS criterion.

A single new-source check does not establish broad generalization/FPR/unique phase causality/scientific PASS. No dynamic deletion, threshold or enlarged sample is implemented. Agent has not executed the notebook or models. Eight successful receiver pins plus G-recorded sentencepiece/ftfy, WanPipeline and both VAE imports are probed; pip-check warnings are retained.

Separate source call budget: {"generation_load": 1, "trajectory": 1, "transformer_conditional": 50, "transformer_unconditional": 50, "native_step": 50, "local_control": 25, "payload_gradient": 25, "pilot_gradient": 0, "wan_vae_load": 1, "wan_decode": 1, "raster_save": 1}

Separate M05/media budget: {"source_read": 1, "framewise_vae_load": 1, "framewise_writer_encode": 1, "writer_sync": 1, "framewise_writer_decode": 1, "raster_save": 1, "mp4_save": 1, "mp4_probe": 1, "mp4_readback": 1}

UNPUBLISHED DRAFT: SOURCE_SHA=None; publish reviewed source and bind its immutable SHA before Run all.

In [ ]:
SOURCE_SHA = None
FIXED = {'observations': 4, 'sync_reads': 8, 'sync_candidate_scores': 392, 'sync_candidate_tubelet_rows': 9456, 'logical_payload_reads': 16, 'logical_detailed_votes': 506880, 'logical_time_bit_rows': 16896, 'logical_final_bit_rows': 512}
from pathlib import Path
import datetime,hashlib,json,os,signal,subprocess,sys,time,traceback
if SOURCE_SHA is None:
    raise RuntimeError('UNPUBLISHED_DRAFT: publish reviewed source and rebuild with its immutable SHA before Run all')
if not isinstance(SOURCE_SHA,str) or len(SOURCE_SHA)!=40:
    raise RuntimeError('immutable 40-character source SHA required')
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Trajectory-Receiver-Independent-Source-Milestone1B-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-TRAJECTORY-RECEIVER-INDEPENDENT-SOURCE-MILESTONE1B-V1-'+STAMP)
PYTHON=sys.executable
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    captured_traceback=''.join(traceback.format_exception(type(exc),exc,exc.__traceback__))
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=f'{type(exc).__name__}: {exc}',traceback=captured_traceback,fixed_denominator=FIXED))
def logged(command,stage,cwd=None,check=True):
    child=None;code=None;primary=None;primary_tb=None;cleanup_errors=[]
    def retain_cleanup_error(label,exc):
        nonlocal primary,primary_tb
        if primary is None and not isinstance(exc,Exception):
            primary=exc;primary_tb=exc.__traceback__
        else:
            cleanup_errors.append(label+': '+repr(exc))
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            popen_group={'start_new_session':True} if os.name=='posix' else {}
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,**popen_group)
            try:
                for line in child.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                code=child.wait()
            except BaseException as exc:
                primary=exc;primary_tb=exc.__traceback__
            finally:
                if child is not None:
                    if os.name=='posix':
                        pgid=child.pid
                        try:os.killpg(pgid,signal.SIGTERM)
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('group SIGTERM',exc)
                        try:
                            deadline=time.monotonic()+5.0
                            while time.monotonic()<deadline:
                                try:os.killpg(pgid,0)
                                except ProcessLookupError:break
                                except BaseException as exc:
                                    retain_cleanup_error('group probe',exc);break
                                time.sleep(0.05)
                        except BaseException as exc:
                            retain_cleanup_error('group TERM grace',exc)
                        finally:
                            try:os.killpg(pgid,signal.SIGKILL)
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('group SIGKILL',exc)
                    elif child.poll() is None:
                        try:child.terminate()
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('terminate',exc)
                    try:
                        try:code=child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            try:child.kill()
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('kill',exc)
                            try:code=child.wait(timeout=10)
                            except BaseException as exc:retain_cleanup_error('wait after kill',exc)
                        except BaseException as exc:retain_cleanup_error('wait',exc)
                    finally:
                        if child.stdout is not None:
                            try:child.stdout.close()
                            except BaseException as exc:retain_cleanup_error('stdout close',exc)
            if primary is None and not cleanup_errors:
                log.write('EXIT '+str(code)+'\n');log.flush()
    except BaseException as wrapper_error:
        if primary is None:
            primary=wrapper_error;primary_tb=wrapper_error.__traceback__
        elif wrapper_error is not primary:
            cleanup_errors.append('log wrapper/close: '+repr(wrapper_error))
    if primary is None and cleanup_errors:
        primary=RuntimeError('process cleanup failed: '+'; '.join(cleanup_errors))
        primary_tb=primary.__traceback__
    if primary is None and check and code:
        primary=subprocess.CalledProcessError(code,command)
        primary_tb=primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary,'add_note'):
            primary.add_note('secondary cleanup errors: '+'; '.join(cleanup_errors))
        try:failed(stage,primary)
        except BaseException as record_error:
            if hasattr(primary,'add_note'):primary.add_note('failure record error: '+repr(record_error))
        raise primary.with_traceback(primary_tb)
    return code
write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED))


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    import shutil
    if any(shutil.which(name) is None for name in ('ffmpeg','ffprobe')):
        logged(['apt-get','update'],'MEDIA_TOOL_UPDATE',check=False)
        logged(['apt-get','install','-y','ffmpeg'],'MEDIA_TOOL_REPAIR',check=False)
    probe="import importlib.metadata as m; pins={'torch': '2.11.0', 'diffusers': '0.39.0', 'transformers': '4.57.6', 'numpy': '2.1.3', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2', 'sentencepiece': '0.2.2', 'ftfy': '6.3.1'}; actual={k:m.version(k) for k in pins}; assert all(actual[k].split('+')[0]==v for k,v in pins.items()), actual; from diffusers import AutoencoderKL, AutoencoderKLWan, WanPipeline; import torch, sentencepiece, ftfy; print(actual)"
    if logged([PYTHON,'-c',probe],'RECEIVER_DEPENDENCY_PROBE',check=False):
        logged([PYTHON,'-m','pip','install','torch==2.11.0','diffusers==0.39.0','transformers==4.57.6','numpy==2.1.3','accelerate==1.15.0','safetensors==0.8.0','huggingface-hub==0.36.2','tokenizers==0.22.2','sentencepiece==0.2.2','ftfy==6.3.1'],'RECEIVER_DEPENDENCY_REPAIR')
        logged([PYTHON,'-c',probe],'RECEIVER_DEPENDENCY_REPROBE')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_COMPLETE',source_sha=actual,fixed_denominator=FIXED,dependency_check_returncode=dependency_code))
except Exception as exc:
    try:failed('SOURCE_OR_ENVIRONMENT',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:pass
finally:token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_trajectory_receiver_independent_source_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; fixed denominator retained')
except Exception as exc:
    try:failed('RUNNER',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'],'Calls:',result['calls'])
print('Physical plan:',result['physical_plan']['planned_calls'],'logical aliases are not independent evidence')
print('Actual environment:',result['environment'])
print('Seals:',result.get('blind_sync_sha256'),result.get('physical_plan_sha256'),result.get('blind_payload_sha256'))
for sid,row in sorted(result['estimates'].items()):
    print('Blind estimate:',sid,row)
for sid,row in sorted(result['sync_posthoc'].items()):
    print('Postseal sync:',sid,'key=',row.get('key_role'),'truth_offset=',row.get('truth_offset'),
          'estimated_offset=',row.get('estimated_offset'),'offset_correct=',row.get('offset_correct'),
          'phase_correct=',row.get('phase_correct'),'geometry_only=',row.get('geometry_only'))
for sid,row in sorted(result['payload_posthoc'].items()):
    values=[x['signed_normalized_margin'] for x in row.get('bit_rows',[]) if 'signed_normalized_margin' in x]
    print(sid,row['status'],'key=',row.get('key_role'),'view=',row.get('view'),'phase=',row.get('phase'),
          'alias=',row.get('alias_of'),'bit_errors=',row.get('bit_errors'),'error_bits=',row.get('error_bits'),'minimum signed margin=',min(values,default=None))
    print('  full detailed evidence:',result['payload_reads'][sid]['detail_path'],row.get('path'))
for sid,row in sorted(result['same_run_differences'].items()):
    print('EST_ALIGN minus BASELINE:',sid,row['status'],'bit error delta=',row.get('bit_error_delta'))
print('Preparation:',result.get('source_preparation'))
print('Generation/M05:',result.get('generated_source'))
print('Preparation workers:',result.get('preparation_workers'))
print('Failures:',result['failures'])
print('Evidence ceiling:',result['evidence_ceiling'])
print('Result:',RESULT_PATH)
